In [1]:
#useful Python libraries
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
#sklearn modules
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from xgboost import XGBClassifier

In [2]:
#load my data
df = pd.read_parquet("my_feature_space.parquet")
df

,objectId,finkclass,mean,weighted_mean,standard_deviation,median,amplitude,beyond_1_std,cusum,inter_percentile_range_10,...,magnitude_percentage_ratio_20_10,maximum_slope,median_absolute_deviation,median_buffer_range_percentage_10,percent_amplitude,mean_variance,anderson_darling_normal,chi2,skew,stetson_K
0,ZTF17aaaadkj,CataclyV*,17.222114,17.174713,0.237890,17.224249,1.171396,0.195322,0.108385,0.481224,...,0.676086,410.613047,0.126921,0.457310,1.825376,0.013813,13.875019,242.808733,-2.438629,0.585522
1,ZTF17aaaagyq,CataclyV*,16.735330,16.498882,0.637000,16.862818,1.910264,0.197590,0.152603,1.436352,...,0.429171,354.654346,0.234251,0.414458,2.552179,0.038063,43.600593,2694.289711,-1.706663,0.709195
2,ZTF17aaaaqna,Unknown,14.323769,14.321861,0.206714,14.224896,0.440336,0.169289,0.109419,0.441297,...,0.723493,95.102274,0.062597,0.344004,0.750938,0.014432,138.605800,238.412293,1.464617,0.796598
3,ZTF17aaaarmr,CataclyV*,16.282178,16.256730,0.240892,16.229107,1.577422,0.129736,0.238448,0.418315,...,0.507220,137.920000,0.074253,0.771527,2.849228,0.014795,61.877870,121.862796,5.046224,0.701406
4,ZTF17aaaazob,CataclyV*,17.864093,17.649710,0.403722,17.859806,2.242684,0.179581,0.082159,0.778325,...,0.627876,696.187235,0.198764,0.561076,3.199834,0.022600,49.381505,620.820245,-2.514259,0.488813
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2054,ZTF26aaajqnu,CataclyV*,16.513851,16.289990,0.722135,16.555887,3.018303,0.117801,0.068421,0.868633,...,0.642155,243.547828,0.209332,0.628272,3.193780,0.043729,30.237867,2277.639225,-1.465526,0.593412
2055,ZTF26aaaombw,Unknown,15.516807,15.516408,0.037666,15.522238,0.106918,0.317241,0.289659,0.100583,...,0.651676,83.947826,0.024834,0.230345,0.113693,0.002427,7.927470,12.038277,-0.567986,0.803067
2056,ZTF26aabsgfq,Unknown,14.029994,14.029941,0.029484,14.032178,0.103438,0.282857,0.198469,0.072323,...,0.627597,29.321250,0.017713,0.320000,0.111945,0.002101,1.771802,7.516016,-0.083897,0.769782
2057,ZTF26aaewgqp,Unknown,14.614267,14.612863,0.066856,14.611416,0.766450,0.051392,0.122444,0.084457,...,0.658268,181.159533,0.022746,0.966809,1.428257,0.004575,76.158691,14.791354,12.549345,0.479608


In [3]:
# 1 = CataclyV (including candidates)
# 0 = everything else

#y_true = df['finkclass'].apply(
 #   lambda x: 1 if 'Cat' in str(x) else 0     #true class title
#).values

In [4]:
# Load saved model
#bestmodel at first
best_model = joblib.load("cv_classifier_best_model_58.pkl")

In [5]:
#feature space
feature_columns = df.drop(columns=['objectId', 'finkclass']).columns #drop non_numeric columns

X = df[feature_columns].values #feature column

In [6]:
y_pred = best_model.predict(X) #predict model
print("Unique predictions:", np.unique(y_pred)) #cross check

Unique predictions: [0 1]


In [7]:
# Check class probabilities for the test set.
# predict_proba returns an array of shape (N_test, 2),
# where N_test is the number of test objects.
# Column 0 : probability of class 0 (negative, unknown)
# Column 1 : probability of class +1 (positive, cvs)
probs = best_model.predict_proba(X)
print('probs',probs)

probs [[0.00545526 0.99454474]
 [0.35834718 0.6416528 ]
 [0.9948584  0.0051416 ]
 ...
 [0.9334283  0.06657171]
 [0.9661173  0.03388267]
 [0.99673057 0.00326943]]


In [8]:
# CV probability
cv_prob = probs[:,1]

# create dataframe for attaching objectid
prob_df = pd.DataFrame({
    "objectId": df["objectId"],
    "cv_probability": cv_prob
})
#check output
print(prob_df.head())

       objectId  cv_probability
0  ZTF17aaaadkj        0.994545
1  ZTF17aaaagyq        0.641653
2  ZTF17aaaaqna        0.005142
3  ZTF17aaaarmr        0.080903
4  ZTF17aaaazob        0.970930


In [9]:
# CV probability with finkclass_column_atached
cv_prob = probs[:,1]

# create dataframe with required columns
prob_df = pd.DataFrame({
    "objectId": df["objectId"],
    "finkclass": df["finkclass"],
    "cv_probability": cv_prob
})

# check
print(prob_df.head())

       objectId  finkclass  cv_probability
0  ZTF17aaaadkj  CataclyV*        0.994545
1  ZTF17aaaagyq  CataclyV*        0.641653
2  ZTF17aaaaqna    Unknown        0.005142
3  ZTF17aaaarmr  CataclyV*        0.080903
4  ZTF17aaaazob  CataclyV*        0.970930


In [10]:
#saving into file
prob_df.to_csv("best_model_prediction_RS_58.csv", index=False)

In [11]:
prob_df

,objectId,finkclass,cv_probability
0,ZTF17aaaadkj,CataclyV*,0.994545
1,ZTF17aaaagyq,CataclyV*,0.641653
2,ZTF17aaaaqna,Unknown,0.005142
3,ZTF17aaaarmr,CataclyV*,0.080903
4,ZTF17aaaazob,CataclyV*,0.970930
...,...,...,...
2054,ZTF26aaajqnu,CataclyV*,0.945626
2055,ZTF26aaaombw,Unknown,0.048755
2056,ZTF26aabsgfq,Unknown,0.066572
2057,ZTF26aaewgqp,Unknown,0.033883


In [12]:
#probablity distribution checking
high_prob = prob_df[prob_df["cv_probability"] > 0.9]
print("Number of objects with CV probability > 0.9:", len(high_prob))

Number of objects with CV probability > 0.9: 651


In [13]:
#saving into parquet
high_prob.to_csv("best_model_prediction_cv_candidates_above_0.9_RS_58.csv", index=False)

In [14]:
high_prob

,objectId,finkclass,cv_probability
0,ZTF17aaaadkj,CataclyV*,0.994545
4,ZTF17aaaazob,CataclyV*,0.970930
6,ZTF17aaabavb,CataclyV*,0.998017
7,ZTF17aaabfay,CataclyV*,0.907813
9,ZTF17aaabpmv,CataclyV*,0.979166
...,...,...,...
2035,ZTF25aaguzft,CataclyV*,0.995758
2036,ZTF25aagvjel,CataclyV*,0.992647
2045,ZTF25abungcd,CataclyV*,0.995097
2051,ZTF26aaaelef,CataclyV*,0.997970


In [15]:
#now for previously saved model
# Load saved model
model = joblib.load("cv_classifier_xgb_boost_biggie_set_58.pkl")

In [16]:
y_pred_1 = model.predict(X) #predict model

In [17]:
y_pred_1

array([1, 0, 0, ..., 0, 0, 0], shape=(2059,))

In [18]:
# Check class probabilities for the test set.
# predict_proba returns an array of shape (N_test, 2),
# where N_test is the number of test objects.
# Column 0 : probability of class 0 (negative, unknown)
# Column 1 : probability of class +1 (positive, cvs)
probs_1 = model.predict_proba(X)
print('probs_1',probs_1)

probs_1 [[0.13235742 0.8676426 ]
 [0.82479906 0.17520097]
 [0.9435169  0.05648309]
 ...
 [0.8097979  0.1902021 ]
 [0.97838444 0.02161557]
 [0.98165476 0.01834524]]


In [19]:
# CV probability
cv_prob_1 = probs_1[:,1]

# create dataframe for attaching objectid
prob_df_1 = pd.DataFrame({
    "objectId": df["objectId"],
    "finkclass": df["finkclass"],
    "cv_probability": cv_prob_1
})
#check output
print(prob_df_1.head())

       objectId  finkclass  cv_probability
0  ZTF17aaaadkj  CataclyV*        0.867643
1  ZTF17aaaagyq  CataclyV*        0.175201
2  ZTF17aaaaqna    Unknown        0.056483
3  ZTF17aaaarmr  CataclyV*        0.113618
4  ZTF17aaaazob  CataclyV*        0.701850


In [20]:
#saving into parquet
prob_df_1.to_csv("class prediction for previousy saved model_RS_58.csv", index=False)

In [21]:
prob_df_1

,objectId,finkclass,cv_probability
0,ZTF17aaaadkj,CataclyV*,0.867643
1,ZTF17aaaagyq,CataclyV*,0.175201
2,ZTF17aaaaqna,Unknown,0.056483
3,ZTF17aaaarmr,CataclyV*,0.113618
4,ZTF17aaaazob,CataclyV*,0.701850
...,...,...,...
2054,ZTF26aaajqnu,CataclyV*,0.891124
2055,ZTF26aaaombw,Unknown,0.040914
2056,ZTF26aabsgfq,Unknown,0.190202
2057,ZTF26aaewgqp,Unknown,0.021616


In [22]:
#probablity distribution checking
high_prob_1 = prob_df_1[prob_df_1["cv_probability"] > 0.9]
print("Number of objects with CV probability > 0.9:", len(high_prob_1))

Number of objects with CV probability > 0.9: 280


In [23]:
#saving into parquet
high_prob_1.to_csv("cv_candidates_above_0.9_for_previously_saved_model_RS_58.csv", index=False)

In [24]:
high_prob_1

,objectId,finkclass,cv_probability
6,ZTF17aaabavb,CataclyV*,0.915050
13,ZTF17aaadasj,CataclyV*,0.969722
18,ZTF17aaaehqt,CataclyV*,0.925015
23,ZTF17aaaeslm,CataclyV*,0.956481
31,ZTF17aaagccm,CataclyV*,0.918923
...,...,...,...
2027,ZTF25aaapxey,CataclyV*,0.984316
2034,ZTF25aagjobe,Unknown,0.911205
2035,ZTF25aaguzft,CataclyV*,0.951666
2036,ZTF25aagvjel,CataclyV*,0.930029


In [25]:
#compare b/w them
y_pred_1 = best_model.predict(X)
y_pred_2 = model.predict(X)

In [26]:
##accuracy score
#print("best_model:", accuracy_score(y_true, y_pred_1))
#print("model:", accuracy_score(y_true, y_pred_2))

In [27]:
#find the mismatch
#load high probability files
high_prob_best = pd.read_csv(
    "best_model_prediction_cv_candidates_above_0.9_RS_58.csv"
)

high_prob_old = pd.read_csv(
    "cv_candidates_above_0.9_for_previously_saved_model_RS_58.csv"
)

In [28]:
#mismatches
cv_best = set(high_prob_best["objectId"])
cv_old = set(high_prob_old["objectId"])

only_in_best = cv_best - cv_old
only_in_old = cv_old - cv_best

print("Only in best model:", len(only_in_best))
print("Only in old model:", len(only_in_old))

Only in best model: 386
Only in old model: 15


In [29]:
#extra objects of new model
extra_best = high_prob_best[
    high_prob_best["objectId"].isin(only_in_best)
]
#extra objects of old model
extra_old = high_prob_old[
    high_prob_old["objectId"].isin(only_in_old)
]
#print
print(extra_best)
print(extra_old)

         objectId  finkclass  cv_probability
0    ZTF17aaaadkj  CataclyV*        0.994545
1    ZTF17aaaazob  CataclyV*        0.970930
3    ZTF17aaabfay  CataclyV*        0.907814
4    ZTF17aaabpmv  CataclyV*        0.979166
5    ZTF17aaabwtm  CataclyV*        0.929706
..            ...        ...             ...
638  ZTF24abbtdws  CataclyV*        0.993280
640  ZTF24abxhwsr    Unknown        0.986436
644  ZTF25aabykmd    Unknown        0.939510
649  ZTF26aaaelef  CataclyV*        0.997970
650  ZTF26aaajqnu  CataclyV*        0.945627

[386 rows x 3 columns]
         objectId  finkclass  cv_probability
18   ZTF17aaatvny  CataclyV*        0.918938
28   ZTF17aabrzzd  CataclyV*        0.930175
42   ZTF17aacqkto  CataclyV*        0.944975
67   ZTF18aaeuivv  CataclyV*        0.932809
102  ZTF18abomvcg  CataclyV*        0.921971
120  ZTF18abvllnd  CataclyV*        0.938741
135  ZTF18acgplgw  CataclyV*        0.950916
152  ZTF18acufqmy  CataclyV*        0.954055
191  ZTF19aaaotxx    Unknown   

In [30]:
#coomon elements
common_ids = cv_best.intersection(cv_old)

print("Common CV candidates:", len(common_ids))

Common CV candidates: 265


In [31]:
#change in probability
common_best = high_prob_best[
    high_prob_best["objectId"].isin(common_ids)
]

common_old = high_prob_old[
    high_prob_old["objectId"].isin(common_ids)
]

merged = common_best.merge(
    common_old,
    on="objectId",
    suffixes=("_best", "_old")
)

merged["prob_diff"] = abs(
    merged["cv_probability_best"]
    - merged["cv_probability_old"]
)

In [32]:
#show
merged.sort_values(
    "prob_diff",
    ascending=True
)

,objectId,finkclass_best,cv_probability_best,finkclass_old,cv_probability_old,prob_diff
95,ZTF18abmolxg,CataclyV*,0.911075,CataclyV*,0.911375,0.000300
111,ZTF18abuapyt,Unknown,0.956515,Unknown,0.954677,0.001838
32,ZTF17aabwnkw,CataclyV*,0.960986,CataclyV*,0.958851,0.002135
58,ZTF18aabgnmu,CataclyV*,0.914940,CataclyV*,0.911496,0.003444
143,ZTF18acuajcr,CataclyV*,0.986752,CataclyV*,0.981483,0.005270
...,...,...,...,...,...,...
184,ZTF19aabgfjp,CataclyV*,0.994123,CataclyV*,0.904277,0.089846
64,ZTF18aaglmmw,Unknown,0.991447,Unknown,0.900455,0.090992
29,ZTF17aabtdus,CataclyV*,0.995552,CataclyV*,0.903371,0.092182
104,ZTF18absgzlq,CataclyV*,0.992454,CataclyV*,0.900113,0.092340
